# Lab 7

## Part A

In [ ]:
import nltk

In [ ]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
from nltk.corpus import wordnet as wn

In [ ]:
wn.synsets('sink')

[Synset('sink.n.01'),
 Synset('sink.n.02'),
 Synset('sinkhole.n.01'),
 Synset('cesspool.n.01'),
 Synset('sink.v.01'),
 Synset('sink.v.02'),
 Synset('sink.v.03'),
 Synset('sink.v.04'),
 Synset('sink.v.05'),
 Synset('dip.v.08'),
 Synset('slump.v.03'),
 Synset('slump.v.02'),
 Synset('bury.v.05')]

In [ ]:
print(len(wn.synsets('sink')))

13


The number of synsets for the english word 'sink' is 13. From listing out the synsets of 'sink', we see that it has two different parts of speech 'n' for noun and 'v' for verb.

In [ ]:
sink = wn.synset('sink.n.01')

In [ ]:
sink

Synset('sink.n.01')

In [ ]:
sink.hypernyms()

[Synset('plumbing_fixture.n.01')]

plumbing fixture is the direct hypernyn of the most common noun sense of sink

In [ ]:
drink = wn.synset('drink.v.01')

In [ ]:
drink.hyponyms()

[Synset('gulp.v.01'),
 Synset('guzzle.v.01'),
 Synset('toss_off.v.02'),
 Synset('drain_the_cup.v.01'),
 Synset('guggle.v.03'),
 Synset('suck.v.01'),
 Synset('lap.v.04'),
 Synset('sip.v.01'),
 Synset('swill.v.02')]

The direct troponyms of drink are shown above

In [ ]:
dog = wn.synset('dog.n.01')

In [ ]:
insect = wn.synset('insect.n.01')

In [ ]:
dog.lowest_common_hypernyms(insect)

[Synset('animal.n.01')]

The closest common ancestor in the IS-A hierarchy for dog.n.01 and insect.n.01 is animal.n.01

In [ ]:
astronaut = wn.synset('astronaut.n.01')

In [ ]:
astronaut.instance_hyponyms()

[Synset('gagarin.n.01'),
 Synset('armstrong.n.01'),
 Synset('glenn.n.01'),
 Synset('shepard.n.02'),
 Synset('tereshkova.n.01')]

In [ ]:
len(astronaut.instance_hyponyms())

5

There are 5 named astronauts which are instances of astronaut.n.01

## Part B

In [ ]:
import re

In [ ]:
from google.colab import drive
gdrive_mount = '/content/drive'
drive.mount(gdrive_mount, force_remount=True)

Mounted at /content/drive


In [ ]:
labdir = '/content/drive/My Drive/Teaching/605.646/Fall 2025/lab7/'
%cd "$labdir"

/content/drive/My Drive/Teaching/605.646/Fall 2025/lab7


In [ ]:
with open('wiki.train.txt','r',encoding='utf-8') as f:
  trainLines = f.readlines()

In [ ]:
trainLines[0:10]

['sandpaper\tabrasive\tIt is commonly used as an abrasive, on everything from sandpaper to large machines used in machining metals, plastics, and wood.\n',
 'bag\taccessory\t- Accessory bag with the following items.\n',
 'bag\taccessory\tWhen Julie feels unable to bide her time for a particular fashion accessory, such as a Hermès Birkin bag, for which there is a three-year wait, she shoplifts from the family store – “kind of stealing from yourself”.\n',
 "coincidence\taccident\tIn a coincidence, Ambujam's bullock-cart and another car strike each other in a road accident, right in front of the very same police station.\n",
 'coincidence\taccident\tThe role and legitimacy of coincidence has frequently been the topic of heated arguments ever since Ronald A. Knox categorically stated that "no accident must ever help the detective" (Commandment No. 6 in his "Decalogue").\n',
 'concussion\taccident\tDuring an examination at the local hospital the doctor comes to the conclusion  that Cassie o

In [ ]:
trainData = []
for line in trainLines:
  line = line.strip()
  if not line:
    continue
  parts = line.split('\t')
  if len(parts) >= 3:
    hyponym, hypernym, sentence = parts[0], parts[1], parts[2]
    trainData.append((hyponym, hypernym, sentence))

In [ ]:
trainData[0:10]

[('sandpaper',
  'abrasive',
  'It is commonly used as an abrasive, on everything from sandpaper to large machines used in machining metals, plastics, and wood.'),
 ('bag', 'accessory', '- Accessory bag with the following items.'),
 ('bag',
  'accessory',
  'When Julie feels unable to bide her time for a particular fashion accessory, such as a Hermès Birkin bag, for which there is a three-year wait, she shoplifts from the family store – “kind of stealing from yourself”.'),
 ('coincidence',
  'accident',
  "In a coincidence, Ambujam's bullock-cart and another car strike each other in a road accident, right in front of the very same police station."),
 ('coincidence',
  'accident',
  'The role and legitimacy of coincidence has frequently been the topic of heated arguments ever since Ronald A. Knox categorically stated that "no accident must ever help the detective" (Commandment No. 6 in his "Decalogue").'),
 ('concussion',
  'accident',
  'During an examination at the local hospital the 

In [ ]:
pattern1 = re.compile(r'(?P<hypernym>[\w\s]+?)\s+such\s+as\s+(?P<hyponyms>[\w\s,]+(?:\s+(?:and|or)\s+[\w\s]+)?)',re.IGNORECASE)

In [ ]:
pattern2 = re.compile(r'(?P<hypernym>[\w\s]+?)\s*,?\s*including\s+(?P<hyponyms>[\w\s,]+(?:\s+(?:and|or)\s+[\w\s]+)?)', re.IGNORECASE)


In [ ]:
trainSentences = [sent for i, j, sent in trainData]

In [ ]:
def parse_hyponyms(hyponymString):
  hyponymString = re.sub(r'\s+and\s+',',',hyponymString,flags=re.IGNORECASE)
  hyponymString = re.sub(r'\s+or\s+',',',hyponymString,flags=re.IGNORECASE)
  hyponyms = [h.strip() for h in hyponymString.split(',') if h.strip()]
  hyponyms = [h for h in hyponyms if len(h)<50]
  return hyponyms

In [ ]:
patterns = [("such as", pattern1),("including", pattern2)]

In [ ]:
relations = {"such as": [], "including":[]}

for sentence in trainSentences:
  sentence = sentence.strip()
  if not sentence:
    continue

  for pattern_name, pattern in patterns:
    matches = pattern.finditer(sentence)

    for match in matches:
      hypernym= match.group('hypernym').strip()
      hyponyms_str=match.group('hyponyms').strip()
      hypernym= re.sub(r'[,;.!?]+$', '', hypernym).strip()
      hyponyms=parse_hyponyms(hyponyms_str)

      if hypernym and len(hypernym)<30:
        for hyponym in hyponyms:
          if hyponym and len(hyponym)<30:
            relations[pattern_name].append((hyponym,hypernym,sentence))

In [ ]:
len(relations)

2

In [ ]:
relations["such as"][0:20]

[('the Airbus A340',
  'Aircraft',
  'Aircraft such as the Airbus A340 and the Boeing 747-400 use winglets.'),
 ('the Boeing 747',
  'Aircraft',
  'Aircraft such as the Airbus A340 and the Boeing 747-400 use winglets.'),
 ('the locomotive',
  'based vehicles',
  'Consequently, ground-based vehicles such as the locomotive and automobile were replaced by aircraft such as the airplane and the zeppelin as the leading mode of transportation in North America.'),
 ('James Stewart',
  'including Hollywood notables',
  'After the war Connelly and Hayward raised $2,000,000 (in 1946 dollars) from investors, including Hollywood notables such as James Stewart and Darryl Zanuck, to expand Southwest into the airline business, pending government approval.'),
 ('Darryl Zanuck',
  'including Hollywood notables',
  'After the war Connelly and Hayward raised $2,000,000 (in 1946 dollars) from investors, including Hollywood notables such as James Stewart and Darryl Zanuck, to expand Southwest into the airli

From the 'such as' pattern we can see from the above it gets 'the Boeing 747' as a correct hyponym of 'Aircraft'. An incorrect one is 'foxes' and 'bird species'

In [ ]:
relations["including"][0:20]

[('associate Dean of Arts',
  'serving in various positions',
  'He distinguished himself as a teacher and administrator at the University of Waterloo, serving in various positions including associate Dean of Arts.'),
 ('the Senior Tutor',
  'eleven members of the SCR',
  'It comprises the Principal; eleven members of the SCR including the Senior Tutor, the Dean, the College Administrator and the College Residence Officer; and eleven members of the JCR including the JCR executive.'),
 ('the Dean',
  'eleven members of the SCR',
  'It comprises the Principal; eleven members of the SCR including the Senior Tutor, the Dean, the College Administrator and the College Residence Officer; and eleven members of the JCR including the JCR executive.'),
 ('the College Administrator',
  'eleven members of the SCR',
  'It comprises the Principal; eleven members of the SCR including the Senior Tutor, the Dean, the College Administrator and the College Residence Officer; and eleven members of the JCR 

From the 'including' pattern, a correct example is 'the Senior Tutor' and 'eleven members of the SCR'. An incorrect example is the agriculture' and 'based training'. It should be 'land-based training'.

In [ ]:
with open('wiki.test.txt','r',encoding='utf-8') as f:
 testLines=f.readlines()

In [ ]:
testData= []

In [ ]:
for line in testLines:
  line = line.strip()
  if not line:
    continue
  parts = line.split('\t')
  if len(parts) >= 3:
    testData.append((parts[0],parts[1],parts[2]))


In [ ]:
testData[0:10]

[('stillbirth',
  'abortion',
  'Doctors had expected a stillbirth and recommended an abortion, even though illegal in the Phillipines, to protect her life, but she decided not to have one.'),
 ('stillbirth',
  'abortion',
  'Statues of the Boddhisattva Jizo, erected in memory of an abortion, miscarriage, stillbirth, or young childhood death, began appearing at least as early as 1710 at a temple in Yokohama (see religion and abortion).'),
 ('richness',
  'abundance',
  "'Richness' refers to an abundance of such resources."),
 ('richness',
  'abundance',
  'First, the relative abundance of ethynyl is an indication of the carbon-richness of its environment (as opposed to oxygen, which provides an important destruction mechanism).'),
 ('richness',
  'abundance',
  'It considers how such interactions, along with interactions between species and the abiotic environment, affect community structure and species richness, diversity and patterns of abundance.'),
 ('richness',
  'abundance',
  'R

In [ ]:
testSentences = [sent for i, j, sent in testData]

In [ ]:
testRelations = {"such as": [], "including":[]}

for sentence in testSentences:
  sentence = sentence.strip()
  if not sentence:
    continue
  for pattern_name, pattern in patterns:
    matches = pattern.finditer(sentence)

    for match in matches:
      hypernym =match.group('hypernym').strip()
      hyponyms_str= match.group('hyponyms').strip()
      hypernym=re.sub(r'[,;.!?]+$', '', hypernym).strip()
      hyponyms=parse_hyponyms(hyponyms_str)

      if hypernym and len(hypernym)<30:
        for hyponym in hyponyms:
          if hyponym and len(hyponym)<30:
            testRelations[pattern_name].append((hyponym,hypernym,sentence))

In [ ]:
testRelations["such as"][0:10]

[('an Epinephrine pen',
  'Frequently medications',
  'Frequently medications such as an Epinephrine pen or an Antihistamine such as Diphenhydramine (Benadryl) are prescribed by an allergist in case of accidental ingestion.'),
 ('Benadryl',
  'and an antihistamine',
  'In cases of anaphylaxis, emergency medical personnel typically administer epinephrine (available as an autoinjector, such as EpiPen) and an antihistamine such as Benadryl (diphenhydramine).'),
 ('Cézanne',
  'based modernist painters',
  'Under the influence of Chinese artist and art teacher Liu Haisu (1896–1994), Liu admired, and often appropriated the styles of French-based modernist painters such as Cézanne, van Gogh and Matisse.'),
 ('van Gogh',
  'based modernist painters',
  'Under the influence of Chinese artist and art teacher Liu Haisu (1896–1994), Liu admired, and often appropriated the styles of French-based modernist painters such as Cézanne, van Gogh and Matisse.'),
 ('Matisse',
  'based modernist painters',